# HumanAI Detect - Insan Metni Augmentasyonu: On Isleme (Asama 2, Colab GPU)

**Amac:** `human_backtranslated` (3262 ornek) havuzunu Stanza dilbilimsel analiz + BERTurk/DistilBERT capraz-perplexity + GPT2-Turkish token-rank + burstiness ile isler. Ana human havuzuyla AYNI kriterle (30-850 token) calisir -- TOPUP_LABELS'a dahil, kisa-metin kriteriyle KARISTIRILMAMALI.

**Neden Colab:** Yerel CPU'da BERT-tabanli pseudo-perplexity ~200sn/ornek -- 3262 ornek yerelde ~181 saat surer.

**ONEMLI -- once kontrol et:** Yerel kod degisikliklerinin GitHub'a push edildiginden emin ol.


In [ ]:
# 1. GPU kontrolu
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('UYARI: GPU yok! Runtime -> Change runtime type -> GPU (T4 yeterli) secip tekrar dene.')


In [ ]:
# 2. Repo klonla
GITHUB_REPO = 'https://github.com/BurakCANKURT/humanai-detect.git'
!git clone {GITHUB_REPO} /content/humanai_detect
%cd /content/humanai_detect
!git log --oneline -3


In [ ]:
# 3. Bagimliliklari kur
!pip install -e '.[dev,colab]' -q
!pip install bitsandbytes accelerate -q


In [ ]:
# 4. Stanza Turkce modelini indir
import stanza
stanza.download('tr')


In [ ]:
# 5. .env olustur (bu adim icin API key gerekmez)
env_content = """
OPENAI_API_KEY=
ANTHROPIC_API_KEY=
GEMINI_API_KEY=
LLAMA_API_KEY=
LLAMA_ENDPOINT=
""".strip()
with open('.env', 'w') as f:
    f.write(env_content)
print('.env olusturuldu')


In [ ]:
# 6. Kaynak dosyayi yerel makineden yukle: human_backtranslated.jsonl
from pathlib import Path
from google.colab import files

Path('data/raw/human_backtranslated').mkdir(parents=True, exist_ok=True)

uploaded = files.upload()
for name in uploaded:
    Path(name).rename('data/raw/human_backtranslated/human_backtranslated.jsonl')


In [ ]:
# 6b. (SADECE onceki bir Colab oturumundan kaldiysa) kismi interim checkpoint'i geri yukle -- ilk kez calistiriyorsan bu hucreyi ATLA
from pathlib import Path
from google.colab import files

Path('data/interim/human_backtranslated').mkdir(parents=True, exist_ok=True)
print('Onceki oturumdan kalan interim dosyasini sec, yoksa bu hucreyi atla')
uploaded = files.upload()
for name in uploaded:
    Path(name).rename('data/interim/human_backtranslated/human_backtranslated.jsonl')


In [ ]:
# 7a. human_backtranslated on isleme (ana human kriteriyle AYNI, 30-850 token)
!python scripts/preprocess.py --label human_backtranslated


In [ ]:
# 7b. Sonucu indir
from pathlib import Path
from google.colab import files

src = Path('data/interim/human_backtranslated/human_backtranslated.jsonl')
if src.exists():
    n = sum(1 for _ in open(src, encoding='utf-8'))
    print(f'{n} ornek -> indiriliyor')
    files.download(str(src))
else:
    print('UYARI: dosya bulunamadi.')


## Yerel Makineye Aktarim

Indirilen `human_backtranslated.jsonl`'i `data/interim/human_backtranslated/human_backtranslated.jsonl`'e koy, sonra haber ver -- diger tum topup'larla (ana GPT, kisa-GPT, yeniden-humanize-Qwen-kisa) birlikte kanonik dosyalara birlestirip build_features/retrain adimina gecerim.
